## Kernel Perceptron — Binary Classification

### 1. Problem Setup
Assume we have a dataset  

$$
\{(\mathbf{x}_1, y_1), (\mathbf{x}_2, y_2), \dots, (\mathbf{x}_N, y_N)\}
$$  

where  

- $\mathbf{x}_i \in \mathbb{R}^D$ is the feature vector  
- $y_i \in \{-1, +1\}$ is the binary class label  

The goal is to classify each input into one of two classes.

---

### 2. Kernel Model
The kernel perceptron uses a kernel function $K(\mathbf{x}, \mathbf{x}')$ to implicitly map data to a higher-dimensional space.  

The score for an input $\mathbf{x}$ is:

$$
f(\mathbf{x}) = \sum_{i=1}^{N} \alpha_i \, y_i \, K(\mathbf{x}_i, \mathbf{x})
$$

where $\alpha_i$ is incremented when sample $i$ is misclassified.

---

### 3. Prediction Rule
The predicted class is determined by the sign of the score:

$$
\hat{y} = \text{sign}\left( f(\mathbf{x}) \right) =
\begin{cases} 
+1 & f(\mathbf{x}) > 0 \\
-1 & f(\mathbf{x}) \le 0
\end{cases}
$$

---

### 4. Objective Function
The kernel perceptron updates weights only when a sample is misclassified:

$$
\alpha_i \leftarrow \alpha_i + 1 \quad \text{if } \hat{y}_i \neq y_i
$$

---

### 5. Batch Update
For multiple misclassified samples, the update can be written as:

$$
\alpha_{\text{mis}} \leftarrow \alpha_{\text{mis}} + 1
$$

where "mis" is the set of misclassified indices.

---

### 6. Initialization
The alpha values are initialized as:

$$
\alpha_i = 0 \quad \forall i
$$

At this point all predictions are:

$$
\hat{y}_i = 0 \implies \text{sign}(\hat{y}_i) = -1 \text{ (or default class)}
$$

---

### 7. Iterative Optimization
For each epoch:

1. Compute predictions:  

$$
\hat{y}_i = \text{sign} \Big( \sum_j \alpha_j y_j K(\mathbf{x}_j, \mathbf{x}_i) \Big)
$$

2. Find misclassified samples:  

$$
\hat{y}_i \neq y_i
$$

3. Update $\alpha$ for misclassified points:

$$
\alpha_i \leftarrow \alpha_i + 1
$$

---

### 8. Convergence Criterion
The algorithm stops when:

$$
\hat{y}_i = y_i \quad \forall i
$$

i.e., no misclassifications remain.  
This happens when the data is **kernel-separable** in the chosen feature space.

---

### 9. Geometric Interpretation
The kernel perceptron learns a **nonlinear decision boundary** in the original feature space:  

$$
\sum_{i=1}^{N} \alpha_i y_i K(\mathbf{x}_i, \mathbf{x}) = 0
$$

- Points where $f(\mathbf{x}) > 0$ are classified as $+1$  
- Points where $f(\mathbf{x}) < 0$ are classified as $-1$  

The shape of the boundary depends on the **kernel function**.

---

### 10. Effect of Kernel Parameters

- **RBF / Gaussian Kernel:**

$$
K(\mathbf{x}, \mathbf{y}) = \exp\left(-\gamma \|\mathbf{x} - \mathbf{y}\|^2\right)
$$

- $\gamma$ controls locality  
  - Small $\gamma$: very local → flexible, risk of overfitting  
  - Large $\gamma$: smoother boundary  



#### Linear Kernel:

$$
K(\mathbf{x}, \mathbf{y}) = \mathbf{x}^T \mathbf{y}
$$

- No parameters  
- Produces a **linear decision boundary**  



#### Polynomial Kernel:

$$
K(\mathbf{x}, \mathbf{y}) = (\mathbf{x}^T \mathbf{y} + c)^d
$$

- $d$ = degree → controls nonlinearity  
- $c$ = constant → shifts the decision boundary  



#### Sigmoid Kernel:

$$
K(\mathbf{x}, \mathbf{y}) = \tanh(\alpha \, \mathbf{x}^T \mathbf{y} + c)
$$

- $\alpha$ (coef) controls slope  
- $c$ shifts the function  
- Similar to neural network activation  



#### Laplace Kernel:

$$
K(\mathbf{x}, \mathbf{y}) = \exp\left(-\gamma \|\mathbf{x} - \mathbf{y}\|_1\right)
$$

- Uses **L1 distance** instead of L2  
- More robust to outliers  



#### Cosine Kernel:

$$
K(\mathbf{x}, \mathbf{y}) = \frac{\mathbf{x}^T \mathbf{y}}{\|\mathbf{x}\| \, \|\mathbf{y}\|}
$$

- Measures **angle between vectors**  
- Focuses on direction, not magnitude  

---



### 11. Prediction
After training, predictions for new points $\mathbf{x}_{\text{test}}$ are computed as:

$$
\hat{y}_{\text{test}} = \text{sign}\Big( \sum_{i \in SV} \alpha_i y_i K(\mathbf{x}_i, \mathbf{x}_{\text{test}}) \Big)
$$

where the sum is over **support vectors** with $\alpha_i > 0$.

---

### 12. Algorithm Summary
1. Initialize $\alpha_i = 0$ for all training samples.  
2. For each epoch:
   - Compute predictions $f(\mathbf{x}_i)$  
   - Identify misclassified samples  
   - Increment $\alpha_i$ for misclassified samples  
3. Stop if no misclassifications remain.  
4. Keep only support vectors ($\alpha_i > 0$) for predictions.

---

### 13. Final Optimization Perspective
The kernel perceptron implicitly solves:

$$
y_i \sum_{j \in SV} \alpha_j y_j K(\mathbf{x}_j, \mathbf{x}_i) > 0, \quad \forall i
$$

This produces a **nonlinear classifier** that separates two classes using the chosen kernel function.

In [1]:
class KernelPerceptron:
            
    """ 
        Kernel Perceptron Classifier.
    
        Implements the Kernel Perceptron algorithm for binary classification
        using different kernel functions.
    
        Parameters
        ----------
        epochs : int, default=10
        Maximum number of passes over the training data.
        kernel : str, default='rbf'
        Kernel type to use. Supported kernels:
        'rbf', 'linear', 'poly', 'sigmoid', 'laplace', 'cosine'
        gamma : float, default=0.1
        Parameter for RBF and Laplace kernels. Must be > 0.
        constant : float, default=0
        Additive constant for polynomial and sigmoid kernels.
        degree : int, default=1
        Degree of the polynomial kernel.
        coef : float, default=1
        Multiplicative coefficient for the sigmoid kernel.
    """
    
    def __init__(self,epochs=10,kernel='rbf',gamma=0.1 , constant=0,degree=1 , coef=1):
        
        self.epochs = epochs
        
        # Validate kernel
        valid_kernels = ['rbf','linear','poly','sigmoid','laplace','cosine']
        self.kernel = kernel
        if self.kernel not in valid_kernels:
            raise ValueError(f"Invalid kernel '{self.kernel}'. Choose from {valid_kernels}")

        # Validate gamma
        self.gamma = gamma 
        if self.gamma <= 0:
            raise ValueError("gamma must be > 0")
            
        self.constant = constant
        
        self.degree = degree
        if type(self.degree) != int:
            raise ValueError("degree must be integer")
        
        self.coef = coef
        # alpha values
        self.alpha= None
  

    def _kernel_function(self,X_train,X_test):
        """
        Compute the kernel matrix between X_train and X_test.

        Parameters
        ----------
        X_train : ndarray of shape (n_samples_train, n_features)
        X_test : ndarray of shape (n_samples_test, n_features)

        Returns
        -------
        K : ndarray of shape (n_samples_train, n_samples_test)
            Kernel (similarity) matrix.
        """
        from scipy.spatial.distance import cdist

        X_train = np.asarray(X_train)
        X_test = np.asarray(X_test)
        
        if self.kernel == 'rbf':
            # RBF kernel: K(x, y) = exp(-gamma * ||x-y||^2)
            distance_squared = cdist(X_train, X_test, 'sqeuclidean')
            
            K = np.exp(-self.gamma * distance_squared)
            K = np.maximum(K,0)
            return K
            
        elif self.kernel =='linear':
            # Linear kernel: K(x, y) = x^T y
            K = X_train @ X_test.T
            return K
            
        elif self.kernel =='poly':
            # Polynomial kernel: K(x, y) = (x^T y + constant)^degree
            K = (X_train @ X_test.T + self.constant)**self.degree
            return K

        elif self.kernel=='sigmoid':
            # Sigmoid kernel: K(x, y) = tanh(coef * x^T y + constant)
            K = np.tanh(self.coef * (X_train @ X_test.T) + self.constant)
            return K

        elif self.kernel == 'laplace':
            # Laplace kernel: K(x, y) = exp(-gamma * L1_distance(x, y))
            l1_distance = cdist(X_train, X_test, 'cityblock')
            
            K = np.exp(-self.gamma * l1_distance)
            K = np.maximum(K,0)
            return K
            
        elif self.kernel == 'cosine':
            # Cosine similarity kernel: K(x, y) = 1 - cosine_distance(x, y)
            K = 1-cdist(X_train, X_test, 'cosine')
            return K
  

    def fit(self,X,y):
        """
        Fit the Kernel Perceptron model to training data.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)
            Training features.
        y : ndarray of shape (n_samples,)
            Binary class labels (+1 or -1).

        Raises
        ------
        ValueError
            If no support vectors are found (model did not learn anything).
        """
        X = np.asarray(X)
        
        y = np.asarray(y)
        y  = y.reshape(-1)

        if X.ndim==1:
            X = X.reshape(-1,1)

        N = len(X)
        # Ensure 2D array
        self.alpha = np.zeros(N)
        
        # Precompute kernel matrix for training
        K = self._kernel_function(X,X)

        # Training loop
        for epoch in range(self.epochs):
            
            mistakes =0

            for i in range(N):
                # Compute prediction using kernel               
                y_hat = np.sum(self.alpha * y  * K[:,i])

                y_hat_i = 1 if y_hat>0 else -1

                # Update alpha if misclassified
                if y_hat_i != y[i]:
                    self.alpha[i] +=1
                    mistakes +=1

            if mistakes==0:  
                
                print(f'No mistake in epoch : {epoch}')
                break
                
        if np.sum(self.alpha) == 0:
            raise ValueError("Model did not learn anything (no support vectors found)")
        
        # Keep only support vectors
        mask = self.alpha > 0
        self.X_sv = X[mask]
        self.y_sv = y[mask]
        self.alpha_sv = self.alpha[mask]

        return None

        
    def predict(self,X):
        """
        Predict class labels for samples in X.

        Parameters
        ----------
        X : ndarray of shape (n_samples, n_features)

        Returns
        -------
        y_pred : ndarray of shape (n_samples,)
            Predicted class labels (+1 or -1)
        """
        X = np.asarray(X)
        # Ensure 2D array
        if X.ndim==1:
            X = X.reshape(-1,1)
            
        # Compute kernel between support vectors and test data
        K = self._kernel_function(self.X_sv,X)
        
        # Compute prediction
        y_pred = np.sum(self.alpha_sv[:, None] * self.y_sv[:,None] * K,axis=0)

        return np.where(y_pred >0 ,1,-1) 



## 1. Dataset

$$
X =
\begin{bmatrix}
0 & 0 \\
0 & 1 \\
1 & 0 \\
1 & 1
\end{bmatrix}, \quad
y =
\begin{bmatrix}
-1 \\
1 \\
1 \\
-1
\end{bmatrix}
$$



Goal

Evaluate how different kernels behave:

- Linear
- RBF
- Polynomial
- Sigmoid
- Laplace
- Cosine


We compare their ability to learn **nonlinear decision boundaries**.



In [2]:
import numpy as np

np.random.seed(42)

# Generate data
N = 200
X = np.random.uniform(-2, 2, (N, 2))

# Circle boundary
radius = 1.0
y = np.where(np.sum(X**2, axis=1) > radius**2, 1, -1)

# Kernel configurations
kernels = [
    ("linear", {}),
    ("rbf", {"gamma": 2}),
    ("poly", {"degree": 3, "constant": 1}),
    ("sigmoid", {"coef": 0.5, "constant": 0}),
    ("laplace", {"gamma": 2}),
    ("cosine", {})
]

# Train and evaluate
for name, params in kernels:
    print(f"\nKernel: {name.upper()}")
    
    model = KernelPerceptron(epochs=500, kernel=name, **params)
    
    try:
        model.fit(X, y)
        preds = model.predict(X)
        
        accuracy = np.mean(preds == y)
        
        print(f"Accuracy: {accuracy:.4f}")
        
    except Exception as e:
        print("Error:", str(e))


Kernel: LINEAR
Accuracy: 0.4650

Kernel: RBF
No mistake in epoch : 40
Accuracy: 1.0000

Kernel: POLY
No mistake in epoch : 227
Accuracy: 1.0000

Kernel: SIGMOID
Accuracy: 0.4750

Kernel: LAPLACE
No mistake in epoch : 3
Accuracy: 1.0000

Kernel: COSINE
Accuracy: 0.4850


---


## 2. Comparison of Kernels







- **Linear kernel** → simple but limited  
- **RBF kernel** → most flexible and reliable  
- **Polynomial kernel** → interpretable nonlinearities  
- **Laplace kernel** → alternative to RBF  
- **Sigmoid kernel** → unstable, parameter-sensitive  
- **Cosine kernel** → useful for similarity-based tasks  





$$
\text{Kernel choice determines model capacity and decision boundary shape}
$$

- Use **RBF by default**  
- Use **Polynomial for structured nonlinear relationships**  
- Use **Cosine for similarity-based problems**



---

## 3. Results 

### Linear Kernel

- Performs poorly  
- Cannot capture circular structure  

$$
\text{Linear boundary} \neq \text{Circular boundary}
$$



### RBF Kernel

- Performs **very well**  
- Learns smooth circular boundary  

$$
\text{RBF} \Rightarrow \text{Highly flexible nonlinear model}
$$



### Polynomial Kernel

- Works reasonably well (degree ≥ 2)  
- Captures curvature but less flexible than RBF  



### Sigmoid Kernel

- Performance varies  
- Sensitive to parameters  
- Less reliable  



### Laplace Kernel

- Similar performance to RBF  
- Slightly different boundary due to L1 distance  



### Cosine Kernel

- Struggles on this dataset  
- Focuses on **angle**, not distance  



## Final Insights

### Kernel Effect

- Distance-based kernels (RBF, Laplace) → best for geometric shapes  
- Dot-product kernels (Linear, Cosine) → limited for nonlinear structure  

---

